# 05 — Final Architecture and Significance

**Question.** Which architecture wins after parameter selection, and is the improvement over BM25F statistically supported on untouched confirm queries?

This notebook is the only place where all architecture-level confirm results are compared.


## Setup


In [ ]:
from pathlib import Path
import os, sys, json, time, copy, itertools, importlib.util, subprocess

def locate_project(start=Path.cwd()):
    for parent in [start.resolve(), *start.resolve().parents]:
        if (parent / "pyproject.toml").exists() and (parent / "src/avito_retriever").exists():
            return parent
    candidate = Path("/content/avito-retriever")
    if candidate.exists():
        return candidate
    raise FileNotFoundError("Clone/open avito-retriever or change PROJECT_ROOT in this cell")

PROJECT_ROOT = locate_project()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

def module_available(name):
    try:
        return importlib.util.find_spec(name) is not None
    except ModuleNotFoundError:
        return False

IN_COLAB = module_available("google.colab")
if IN_COLAB and os.environ.get("AVITO_AUTO_INSTALL", "1") != "0":
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-e",
        f"{PROJECT_ROOT}[lexical,neural,dev]",
    ])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from avito_retriever.experiments.notebook import resolve_data_dir, result_dir, highlight_best, save_json, save_yaml

DATA_DIR = resolve_data_dir(PROJECT_ROOT)
SEED = 42
np.random.seed(SEED)
print("Project:", PROJECT_ROOT)
print("Data:", DATA_DIR)


In [ ]:
from avito_retriever.evaluation.statistics import compare_paired_runs
from avito_retriever.evaluation.multiple import holm_bonferroni

R = PROJECT_ROOT / "artifacts/notebook_results"
OUT = result_dir(PROJECT_ROOT, "05_final_architecture_significance")
split = pd.read_csv(R / "01_sentencepiece_bm25f_search/selection_split.csv")
paths = {
 "BM25F": R/"01_sentencepiece_bm25f_search/best_bm25f_per_query.parquet",
 "Dense": R/"02_dense_knn_fusion_search/best_dense_per_query.parquet",
 "kNN": R/"02_dense_knn_fusion_search/best_knn_per_query.parquet",
 "BM25F + Dense": R/"02_dense_knn_fusion_search/best_bm25f_dense_per_query.parquet",
 "BM25F + kNN": R/"02_dense_knn_fusion_search/best_bm25f_knn_per_query.parquet",
 "Dense + kNN": R/"02_dense_knn_fusion_search/best_dense_knn_per_query.parquet",
 "BM25F + Dense + kNN": R/"02_dense_knn_fusion_search/best_bm25f_dense_knn_per_query.parquet",
 "Reranked": R/"03_reranker_comparison/best_reranked_per_query.parquet",
 "OCR fusion": R/"04_ocr_ablation/best_ocr_per_query.parquet",
}
systems = {name: pd.read_parquet(path) for name,path in paths.items() if path.exists()}
required = {"BM25F", "Dense", "kNN", "BM25F + Dense", "BM25F + kNN", "Dense + kNN", "BM25F + Dense + kNN"}
assert required <= systems.keys(), "Run notebooks 01 and 02 with the full architecture ablation first"


## Architecture table


In [ ]:
components = {
 "BM25F": (True, False, False, False, False),
 "Dense": (False, True, False, False, False),
 "kNN": (False, False, True, False, False),
 "BM25F + Dense": (True, True, False, False, False),
 "BM25F + kNN": (True, False, True, False, False),
 "Dense + kNN": (False, True, True, False, False),
 "BM25F + Dense + kNN": (True, True, True, False, False),
 "Reranked": (True, True, True, True, False),
 "OCR fusion": (True, True, True, False, True),
}
rows=[]
for name,frame in systems.items():
    joined=frame.merge(split[["query_id","split"]],on="query_id")
    use_bm25f,use_dense,use_knn,use_reranker,use_ocr=components[name]
    rows.append({"architecture":name,"BM25F":use_bm25f,"Dense":use_dense,"kNN":use_knn,
                 "Reranker":use_reranker,"OCR":use_ocr,
                 "tune_map@10":joined.loc[joined.split=="tune","ap@10"].mean(),
                 "confirm_map@10":joined.loc[joined.split=="confirm","ap@10"].mean(),
                 "overall_map@10":joined["ap@10"].mean(),
                 "recall@50":joined["recall@50"].mean()})
architecture_table=pd.DataFrame(rows).sort_values("tune_map@10",ascending=False).reset_index(drop=True)
display(highlight_best(architecture_table,["tune_map@10","confirm_map@10","overall_map@10","recall@50"]))


## Incremental ablation gains


In [ ]:
score_by_name=architecture_table.set_index("architecture")
comparisons=[
 ("Add Dense to BM25F","BM25F","BM25F + Dense"),
 ("Add kNN to BM25F","BM25F","BM25F + kNN"),
 ("Combine Dense and kNN","Dense","Dense + kNN"),
 ("Add kNN to BM25F + Dense","BM25F + Dense","BM25F + Dense + kNN"),
 ("Add Dense to BM25F + kNN","BM25F + kNN","BM25F + Dense + kNN"),
]
if "Reranked" in systems: comparisons.append(("Add reranker","BM25F + Dense + kNN","Reranked"))
if "OCR fusion" in systems: comparisons.append(("Add OCR","BM25F + Dense + kNN","OCR fusion"))
ablation_rows=[]
for experiment,parent,candidate in comparisons:
    ablation_rows.append({"experiment":experiment,"parent":parent,"candidate":candidate,
                          "tune_delta":score_by_name.loc[candidate,"tune_map@10"]-score_by_name.loc[parent,"tune_map@10"],
                          "confirm_delta":score_by_name.loc[candidate,"confirm_map@10"]-score_by_name.loc[parent,"confirm_map@10"]})
ablation_table=pd.DataFrame(ablation_rows)
display(ablation_table.style.format({"tune_delta":"{:+.4f}","confirm_delta":"{:+.4f}"})
        .background_gradient(subset=["tune_delta","confirm_delta"],cmap="RdYlGn",vmin=-.03,vmax=.03))


## Paired confirm-set tests against BM25F


In [ ]:
confirm_ids=set(split.loc[split.split=="confirm","query_id"])
baseline=systems["BM25F"]; baseline=baseline[baseline.query_id.isin(confirm_ids)]
tests=[]
for name,frame in systems.items():
    if name=="BM25F": continue
    result=compare_paired_runs(baseline,frame[frame.query_id.isin(confirm_ids)],metric="ap@10",
                               bootstrap_samples=10000,permutation_samples=10000,seed=SEED)
    tests.append({"architecture":name,**result})
test_table=pd.DataFrame(tests)
test_table["holm_permutation_p"]=holm_bonferroni(test_table.paired_permutation_p.tolist())
display(test_table[["architecture","baseline_mean","candidate_mean","mean_difference","bootstrap_ci",
                    "paired_permutation_p","holm_permutation_p","wins","ties","losses"]])


## Paired tests for incremental additions


In [ ]:
incremental_tests=[]
for experiment,parent,candidate in comparisons:
    result=compare_paired_runs(
        systems[parent][systems[parent].query_id.isin(confirm_ids)],
        systems[candidate][systems[candidate].query_id.isin(confirm_ids)],
        metric="ap@10",bootstrap_samples=10000,permutation_samples=10000,seed=SEED)
    incremental_tests.append({"experiment":experiment,"parent":parent,"candidate":candidate,**result})
incremental_test_table=pd.DataFrame(incremental_tests)
incremental_test_table["holm_permutation_p"]=holm_bonferroni(
    incremental_test_table.paired_permutation_p.tolist())
display(incremental_test_table[["experiment","parent","candidate","mean_difference","bootstrap_ci",
                                "paired_permutation_p","holm_permutation_p","wins","ties","losses"]])


## Distribution diagnostics


In [ ]:
best_name=architecture_table.iloc[0].architecture
best_frame=systems[best_name].set_index("query_id"); base_frame=systems["BM25F"].set_index("query_id")
delta=(best_frame.loc[list(confirm_ids),"ap@10"]-base_frame.loc[list(confirm_ids),"ap@10"])
fig,axes=plt.subplots(1,2,figsize=(12,4))
delta.hist(bins=20,ax=axes[0]); axes[0].axvline(0,color="black"); axes[0].set_title(f"Confirm AP@10 delta: {best_name} - BM25F")
pd.DataFrame({name:frame.set_index("query_id").loc[list(confirm_ids),"ap@10"] for name,frame in systems.items()}).boxplot(ax=axes[1],rot=25)
axes[1].set_title("Confirm per-query AP@10"); plt.tight_layout(); plt.show()


## Human-readable conclusion


In [ ]:
winner=architecture_table.iloc[0].to_dict()
winner_test=test_table.loc[test_table.architecture==winner["architecture"]].iloc[0] if winner["architecture"]!="BM25F" else None
if winner_test is None:
    conclusion="BM25F remained the strongest tune-selected architecture; added complexity was not justified."
else:
    low,high=winner_test.bootstrap_ci
    significant=(winner_test.holm_permutation_p<0.05 and low>0)
    conclusion=(f"Selected architecture: {winner['architecture']}. Confirm MAP@10 = {winner['confirm_map@10']:.4f}; "
                f"delta vs BM25F = {winner_test.mean_difference:+.4f}; 95% bootstrap CI [{low:.4f}, {high:.4f}]. "
                + ("The improvement is statistically supported." if significant else
                   "The improvement is not yet statistically conclusive after correction."))
display(Markdown("### Conclusion\n"+conclusion))


## Save final decision


In [ ]:
architecture_table.to_csv(OUT/"architecture_comparison.csv",index=False)
ablation_table.to_csv(OUT/"architecture_ablation_deltas.csv",index=False)
test_table.to_json(OUT/"significance_tests.json",orient="records",force_ascii=False,indent=2)
incremental_test_table.to_json(OUT/"incremental_significance_tests.json",orient="records",force_ascii=False,indent=2)
ranking_paths = {
 "BM25F": R/"01_sentencepiece_bm25f_search/best_bm25f_rankings.parquet",
 "Dense": R/"02_dense_knn_fusion_search/best_dense_rankings.parquet",
 "kNN": R/"02_dense_knn_fusion_search/best_knn_rankings.parquet",
 "BM25F + Dense": R/"02_dense_knn_fusion_search/best_bm25f_dense_rankings.parquet",
 "BM25F + kNN": R/"02_dense_knn_fusion_search/best_bm25f_knn_rankings.parquet",
 "Dense + kNN": R/"02_dense_knn_fusion_search/best_dense_knn_rankings.parquet",
 "BM25F + Dense + kNN": R/"02_dense_knn_fusion_search/best_bm25f_dense_knn_rankings.parquet",
 "Reranked": R/"03_reranker_comparison/best_reranked_rankings.parquet",
 "OCR fusion": R/"04_ocr_ablation/best_ocr_rankings.parquet",
}
selected={"architecture":winner["architecture"],"ranking_source":str(ranking_paths[winner["architecture"]]),
 "conclusion":conclusion}
save_json(selected,OUT/"final_selected_architecture.json")
print("Saved final decision and statistical evidence")
